<a href="https://colab.research.google.com/github/goitstudent123/numerical_programming_python/blob/main/%D0%94%D0%9712_%D0%93%D0%90%D0%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install and import
!pip install -q plotly==5.20.0
!pip install -q "jupyterlab>=3" "ipywidgets>=7.6"

import os
import zipfile
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from scipy import stats


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.7 MB/s eta 0:00:00


In [2]:
# 2. Download the dataset zip
!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true


--2025-11-02 13:38:09--  https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/raw/refs/heads/main/WorldHappinessReport.zip [following]
--2025-11-02 13:38:09--  https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/raw/refs/heads/main/WorldHappinessReport.zip
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/refs/heads/main/WorldHappinessReport.zip [following]
--2025-11-02 13:38:09--  https://raw.githubusercontent.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/refs/heads/main/WorldHappinessReport.zip
Resolving raw.githubusercontent.com (raw.githubuserco

In [3]:
# 3. Unzip
!unzip -o WorldHappinessReport.zip


Archive:  WorldHappinessReport.zip
  inflating: 2015.csv                
  inflating: 2016.csv                
  inflating: 2017.csv                
  inflating: 2018.csv                
  inflating: 2019.csv                


In [5]:
# 4. Read 2017 csv and inspect basic info
# Why 2017? Because Kaggle + WHR tutorials always use 2017 edition as canonical for intro ))) lol

data = pd.read_csv("2017.csv")

# normalize column names for consistency
data = data.rename(columns={
    'Happiness.Score': 'Happiness.Score',
    'Happiness.Rank': 'Happiness.Rank',
    'Country': 'Country'
})

display(data.head())
print("\nInfo:")
print(data.info())
print("\nDescribe:")
display(data.describe().T)

,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual
0,Norway,1,7.537,7.594445,7.479556,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027
1,Denmark,2,7.522,7.581728,7.462272,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707
2,Iceland,3,7.504,7.622030,7.385970,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715
3,Switzerland,4,7.494,7.561772,7.426227,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716
4,Finland,5,7.469,7.527542,7.410458,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        155 non-null    object 
 1   Happiness.Rank                 155 non-null    int64  
 2   Happiness.Score                155 non-null    float64
 3   Whisker.high                   155 non-null    float64
 4   Whisker.low                    155 non-null    float64
 5   Economy..GDP.per.Capita.       155 non-null    float64
 6   Family                         155 non-null    float64
 7   Health..Life.Expectancy.       155 non-null    float64
 8   Freedom                        155 non-null    float64
 9   Generosity                     155 non-null    float64
 10  Trust..Government.Corruption.  155 non-null    float64
 11  Dystopia.Residual              155 non-null    float64
dtypes: float64(10), int64(1), object(1)
memory 

,count,mean,std,min,25%,50%,75%,max
Happiness.Rank,155.0,78.000000,44.888751,1.000000,39.500000,78.000000,116.500000,155.000000
Happiness.Score,155.0,5.354019,1.131230,2.693000,4.505500,5.279000,6.101500,7.537000
Whisker.high,155.0,5.452326,1.118542,2.864884,4.608172,5.370032,6.194600,7.622030
Whisker.low,155.0,5.255713,1.145030,2.521116,4.374955,5.193152,6.006527,7.479556
Economy..GDP.per.Capita.,155.0,0.984718,0.420793,0.000000,0.663371,1.064578,1.318027,1.870766
Family,155.0,1.188898,0.287263,0.000000,1.042635,1.253918,1.414316,1.610574
Health..Life.Expectancy.,155.0,0.551341,0.237073,0.000000,0.369866,0.606042,0.723008,0.949492
Freedom,155.0,0.408786,0.149997,0.000000,0.303677,0.437454,0.516561,0.658249
Generosity,155.0,0.246883,0.134780,0.000000,0.154106,0.231538,0.323762,0.838075
Trust..Government.Corruption.,155.0,0.123120,0.101661,0.000000,0.057271,0.089848,0.153296,0.464308


In [6]:
# 5. Plot numeric distributions and basic normality hints

numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
fig = px.histogram(data, x=numeric_cols, facet_col_wrap=3, nbins=30, marginal="box", opacity=0.85)
fig.update_layout(title="Numeric Feature Distributions")
fig.show()

# Shapiro-Wilk on up to 5000 samples per column (p-value only)
normality = []
for col in numeric_cols:
    col_series = data[col].dropna()
    if len(col_series) >= 3:
        sample = col_series.sample(min(len(col_series), 5000), random_state=42)
        stat, p = stats.shapiro(sample)
        normality.append({"feature": col, "shapiro_p": p, "skew": sample.skew(), "kurtosis": sample.kurtosis()})
pd_normality = pd.DataFrame(normality).sort_values("shapiro_p", ascending=False)
display(pd_normality)


,feature,shapiro_p,skew,kurtosis
3,Whisker.low,6.279252e-02,0.009116,-0.723334
1,Happiness.Score,5.222746e-02,0.009554,-0.750419
2,Whisker.high,5.075453e-02,0.008410,-0.776360
10,Dystopia.Residual,3.341324e-02,-0.239324,0.690233
4,Economy..GDP.per.Capita.,1.749610e-03,-0.390693,-0.676755
7,Freedom,1.673262e-04,-0.615766,-0.208406
8,Generosity,1.184041e-04,0.898715,1.743408
0,Happiness.Rank,6.077553e-05,0.000000,-1.200000
6,Health..Life.Expectancy.,1.134614e-05,-0.577966,-0.585552
5,Family,4.186380e-08,-1.181100,1.535250


In [7]:
# 6. Select domain-relevant features and compute correlation

# Try to map common column variants across years
def find_col(candidates):
    for c in data.columns:
        key = c.lower().replace(" ", "").replace(".", "").replace("_", "")
        for cand in candidates:
            if key == cand:
                return c
    return None

col_map = {
    "GDP": find_col(["economygdppercapita","gdppercapita","loggedgdppercapita","gdppercapita2021","gdppercapita2017","gdppercapita2016","gdppercapita2015"]),
    "Family": find_col(["family","socialsupport"]),
    "Health": find_col(["healthlifeexpectancy","healthylifeexpectancy"]),
    "Freedom": find_col(["freedom","freedomtomakelifechoices"]),
    "Trust": find_col(["trustgovernmentcorruption","perceptionsofcorruption"]),
    "Generosity": find_col(["generosity"]),
    "Happiness.Score": "Happiness.Score" if "Happiness.Score" in data.columns else find_col(["score","happinessscore"])
}

selected = [v for v in col_map.values() if v is not None]
features_df = data[selected].copy()
display(features_df.head())

corr = features_df.corr(numeric_only=True)
fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_layout(title="Correlation Matrix (Selected Features)")
fig.show()


,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Trust..Government.Corruption.,Generosity,Happiness.Score
0,1.616463,1.533524,0.796667,0.635423,0.315964,0.362012,7.537
1,1.482383,1.551122,0.792566,0.626007,0.400770,0.355280,7.522
2,1.480633,1.610574,0.833552,0.627163,0.153527,0.475540,7.504
3,1.564980,1.516912,0.858131,0.620071,0.367007,0.290549,7.494
4,1.443572,1.540247,0.809158,0.617951,0.382612,0.245483,7.469


In [8]:
# 7. Sorted Pearson correlations vs target
target = "Happiness.Score" if "Happiness.Score" in features_df.columns else None
if target:
    corrs = corr[target].drop(labels=[target], errors="ignore").sort_values(ascending=False)
    display(pd.DataFrame({"feature": corrs.index, "pearson_corr_with_happiness": corrs.values}))
else:
    print("Target 'Happiness.Score' not present in current selection.")


,feature,pearson_corr_with_happiness
0,Economy..GDP.per.Capita.,0.812469
1,Health..Life.Expectancy.,0.781951
2,Family,0.752737
3,Freedom,0.570137
4,Trust..Government.Corruption.,0.429080
5,Generosity,0.155256


In [9]:
# 8. Choropleth map for Happiness.Score
df_map = data.copy()
if "Happiness.Score" not in df_map.columns:
    # If only Rank exists, convert to pseudo score by inverse rank as fallback
    if "Happiness.Rank" in df_map.columns:
        df_map["Happiness.Score"] = (df_map["Happiness.Rank"].max() + 1) - df_map["Happiness.Rank"]
    else:
        raise RuntimeError("No 'Happiness.Score' or 'Happiness.Rank' available for mapping.")

if "Country" not in df_map.columns:
    raise RuntimeError("No 'Country' column found for map.")

fig = px.choropleth(df_map,
                    locations="Country",
                    color="Happiness.Score",
                    locationmode="country names")
fig.update_layout(title="Happiness Index (Selected Year)")
fig.show()


In [10]:
# 9. Scaling using provided function (minmax/std/norm)

def data_scale(data, scaler_type='minmax'):
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.preprocessing import StandardScaler
    from sklearn.preprocessing import Normalizer
    if scaler_type == 'minmax':
        scaler = MinMaxScaler()
    if scaler_type == 'std':
        scaler = StandardScaler()
    if scaler_type == 'norm':
        scaler = Normalizer()

    scaler.fit(data)
    res = scaler.transform(data)
    return res

# choose numeric features only (excluding the target to avoid leakage for clustering experiments if desired)
numeric_for_scaling = features_df.select_dtypes(include=[np.number]).columns.tolist()
X_original = features_df[numeric_for_scaling].dropna().copy()

data_scaled = data_scale(X_original, scaler_type='std')
df_scaled = pd.DataFrame(data_scaled, columns=[X_original.columns])
print(df_scaled.head())


  Economy..GDP.per.Capita.    Family Health..Life.Expectancy.   Freedom  \
0                 1.506188  1.203577                 1.038167  1.515836   
1                 1.186518  1.265036                 1.020812  1.452859   
2                 1.182345  1.472669                 1.194259  1.460590   
3                 1.383442  1.145561                 1.298272  1.413155   
4                 1.093985  1.227057                 1.091026  1.398978   

  Trust..Government.Corruption. Generosity Happiness.Score  
0                      1.903084   0.856964        1.935996  
1                      2.739998   0.806856        1.922693  
2                      0.300066   1.702013        1.906730  
3                      2.406809   0.325028        1.897861  
4                      2.560800  -0.010426        1.875689  


In [11]:
# 10. Statistics comparison
print("Original stats:")
display(X_original.describe().T)

print("\nScaled stats (StandardScaler):")
display(df_scaled.describe().T)


Original stats:


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0,0.984718,0.420793,0.000,0.663371,1.064578,1.318027,1.870766
Family,155.0,1.188898,0.287263,0.000,1.042635,1.253918,1.414316,1.610574
Health..Life.Expectancy.,155.0,0.551341,0.237073,0.000,0.369866,0.606042,0.723008,0.949492
Freedom,155.0,0.408786,0.149997,0.000,0.303677,0.437454,0.516561,0.658249
Trust..Government.Corruption.,155.0,0.123120,0.101661,0.000,0.057271,0.089848,0.153296,0.464308
Generosity,155.0,0.246883,0.134780,0.000,0.154106,0.231538,0.323762,0.838075
Happiness.Score,155.0,5.354019,1.131230,2.693,4.505500,5.279000,6.101500,7.537000



Scaled stats (StandardScaler):


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0,-2.750488e-16,1.003241,-2.347736,-0.766147,0.190399,0.794666,2.112488
Family,155.0,3.667317e-16,1.003241,-4.152125,-0.510811,0.227076,0.787253,1.472669
Health..Life.Expectancy.,155.0,-2.292073e-16,1.003241,-2.333157,-0.767962,0.231482,0.726457,1.684893
Freedom,155.0,2.750488e-16,1.003241,-2.734123,-0.703009,0.191745,0.720845,1.668506
Trust..Government.Corruption.,155.0,-9.168293e-17,1.003241,-1.215016,-0.649839,-0.328353,0.297794,3.367022
Generosity,155.0,1.203339e-16,1.003241,-1.837684,-0.690591,-0.114221,0.572250,4.400552
Happiness.Score,155.0,-2.750488e-16,1.003241,-2.359949,-0.752517,-0.066532,0.662910,1.935996


In [12]:
# 11. GaussianMixture clustering
k = 3
gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42)
labels = gmm.fit_predict(df_scaled)
clustered = X_original.copy()
clustered["cluster"] = labels
clustered["Country"] = data.loc[X_original.index, "Country"].values
if "Happiness.Score" in data.columns:
    clustered["Happiness.Score"] = data.loc[X_original.index, "Happiness.Score"].values
display(clustered.head())

if len(set(labels)) > 1:
    sil = silhouette_score(df_scaled, labels)
    print(f"Silhouette score (k={k}): {sil:.3f}")
else:
    print("Silhouette score not defined for a single cluster.")


,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Trust..Government.Corruption.,Generosity,Happiness.Score,cluster,Country
0,1.616463,1.533524,0.796667,0.635423,0.315964,0.362012,7.537,0,Norway
1,1.482383,1.551122,0.792566,0.626007,0.400770,0.355280,7.522,0,Denmark
2,1.480633,1.610574,0.833552,0.627163,0.153527,0.475540,7.504,0,Iceland
3,1.564980,1.516912,0.858131,0.620071,0.367007,0.290549,7.494,0,Switzerland
4,1.443572,1.540247,0.809158,0.617951,0.382612,0.245483,7.469,0,Finland


Silhouette score (k=3): 0.279


In [13]:
# 12. Choropleth of clusters
map_df = clustered[["Country", "cluster"]].copy().dropna()
map_df["cluster_str"] = map_df["cluster"].astype(str)

fig = px.choropleth(map_df,
                    locations="Country",
                    color="cluster_str",
                    locationmode="country names",
                    color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(title="Country Cluster Assignment (GMM)")
fig.show()


In [14]:
# 13. Evaluate different feature subsets and compare clustering stability and quality

def run_gmm_on_features(features, n=3):
    X = features_df[features].dropna()
    idx = X.index
    Xs = pd.DataFrame(data_scale(X, scaler_type='std'), columns=features, index=idx)
    gmm = GaussianMixture(n_components=n, random_state=42)
    labs = gmm.fit_predict(Xs)
    sil = silhouette_score(Xs, labs) if len(set(labs)) > 1 else np.nan
    ref = None
    if "Happiness.Score" in data.columns:
        ref = data.loc[idx, "Happiness.Score"]
    return idx, labs, sil, ref

# Define several subsets
available = set(features_df.columns)
subset_all = [c for c in ["GDP","Family","Health","Freedom","Trust","Generosity","Happiness.Score"] if col_map.get(c) is not None]
subset_all = [col_map[k] if k in col_map and col_map[k] is not None else k for k in ["GDP","Family","Health","Freedom","Trust","Generosity"] if col_map.get(k) is not None]

subsets = {
    "all_econ_social": subset_all,
    "econ_only": [col_map["GDP"]] if col_map.get("GDP") else [],
    "econ_health_freedom": [c for k,c in col_map.items() if k in ["GDP","Health","Freedom"] and c is not None],
    "social_block": [c for k,c in col_map.items() if k in ["Family","Generosity","Trust"] and c is not None],
}

results = []
basel_idx, basel_labs, basel_sil, basel_ref = run_gmm_on_features(subsets["all_econ_social"])
for name, feats in subsets.items():
    if not feats:
        continue
    idx, labs, sil, ref = run_gmm_on_features(feats)
    # Align to baseline index for comparability if possible
    common = basel_idx.intersection(idx)
    ami = adjusted_mutual_info_score(basel_labs[np.isin(basel_idx, common)], labs[np.isin(idx, common)]) if len(common) > 0 else np.nan
    results.append({"subset": name, "features": feats, "silhouette": sil, "AMI_vs_all_econ_social": ami, "n_features": len(feats)})

df_subset_results = pd.DataFrame(results).sort_values(by=["silhouette"], ascending=False)
display(df_subset_results)


,subset,features,silhouette,AMI_vs_all_econ_social,n_features
1,econ_only,[Economy..GDP.per.Capita.],0.577332,0.374266,1
3,social_block,"[Family, Trust..Government.Corruption., Genero...",0.337690,0.440261,3
2,econ_health_freedom,"[Economy..GDP.per.Capita., Health..Life.Expect...",0.324321,0.515228,3
0,all_econ_social,"[Economy..GDP.per.Capita., Family, Health..Lif...",0.268953,1.000000,6


In [16]:
# 14. Compute statistical alignment of clusters with Happiness.Score / Rank

out = {}

# Use the clustering built in step 11 on df_scaled with k=3
c_labels = labels
idx = X_original.index

# Prepare target
if "Happiness.Score" in data.columns:
    target_series = data.loc[idx, "Happiness.Score"]
    # ANOVA across clusters
    groups = [target_series[c_labels == c].dropna() for c in np.unique(c_labels)]
    if len(groups) >= 2 and all(len(g) > 1 for g in groups):
        f_stat, p_val = stats.f_oneway(*groups)
    else:
        f_stat, p_val = np.nan, np.nan
    # Eta-squared as effect size
    grand_mean = target_series.mean()
    ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in groups)
    ss_total = ((target_series - grand_mean)**2).sum()
    eta_sq = ss_between/ss_total if ss_total > 0 else np.nan

    # Rank-based check (quantile bins)
    qbins = pd.qcut(target_series, q=3, labels=["low","mid","high"])
    ami = adjusted_mutual_info_score(qbins.astype(str), c_labels.astype(str))

    # Per-cluster summary
    per_cluster = (pd.DataFrame({
        "cluster": c_labels,
        "score": target_series.values
    }).groupby("cluster")
      .agg(n=("score","size"), mean_score=("score","mean"), std_score=("score","std"))
      .sort_values("mean_score", ascending=False))

    out["anova_F"] = f_stat
    out["anova_p"] = p_val
    out["eta_squared"] = eta_sq
    out["AMI_clusters_vs_score_tertiles"] = ami
    out["per_cluster_stats"] = per_cluster

    print("=== Alignment with Happiness.Score ===")
    print(f"ANOVA F={f_stat:.4f}, p={p_val:.6f}")
    print(f"Eta squared (effect size) = {eta_sq:.4f}")
    print(f"AMI(clusters vs score tertiles) = {ami:.4f}")
    display(per_cluster)
else:
    # Fallback to Rank if Score not present
    if "Happiness.Rank" in data.columns:
        target_series = data.loc[idx, "Happiness.Rank"]
        qbins = pd.qcut(target_series, q=3, labels=["top","mid","bottom"])
        ami = adjusted_mutual_info_score(qbins.astype(str), c_labels.astype(str))
        per_cluster = (pd.DataFrame({
            "cluster": c_labels,
            "rank": target_series.values
        }).groupby("cluster")
          .agg(n=("rank","size"), mean_rank=("rank","mean"), std_rank=("rank","std"))
          .sort_values("mean_rank", ascending=True))
        out["AMI_clusters_vs_rank_tertiles"] = ami
        out["per_cluster_stats"] = per_cluster
        print("=== Alignment with Happiness.Rank (lower is better) ===")
        print(f"AMI(clusters vs rank tertiles) = {ami:.4f}")
        display(per_cluster)
    else:
        print("No target column available.")


=== Alignment with Happiness.Score ===
ANOVA F=147.7651, p=0.000000
Eta squared (effect size) = 0.6604
AMI(clusters vs score tertiles) = 0.4136


,n,mean_score,std_score
cluster,,,
0,26,6.876692,0.548244
2,79,5.579215,0.705971
1,50,4.206420,0.647514


# Conclusions

* ANOVA p-value is extremely small (≈ 0), which means the cluster means differ statistically significantly.
* Effect size η² ≈ 0.66 — this is a very strong effect size.
→ clusters explain ~66% of the variance in happiness score.
* AMI ≈ 0.41 is moderate — this means clustering is not perfect but it is clearly aligned with happiness level ranking.
* per cluster means show clean monotonic grouping:
  * cluster 0 → high happiness (~6.88)
  * cluster 2 → mid happiness (~5.58)
  * cluster 1 → low happiness (~4.21)

Unsupervised clustering discovered essentially the same natural stratification as the actual Happiness Score itself.